In [1]:
#Loading in Packages and Data
##################################################################

#Importing Packages
import numpy as np
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
# import matplotlib.gridspec as gridspec
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
netCDF=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
true_time=netCDF['time']
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***
times=netCDF['time'].values/(1e9 * 60); times=times.astype(float);

#Restricts the timesteps of the data from timesteps0 to 140
data=netCDF.isel(time=np.arange(0,140+1))
parcel=parcel.isel(time=np.arange(0,140+1))
res='1km'

# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# parcel=parcel.isel(time=np.arange(0,400+1))
# res='250m'

In [2]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [70]:
# Reading Back Data Later
##################################################################
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
with h5py.File(dir2+'lagrangian_binary_array.h5', 'r') as f:
    # Load the dataset by its name
    A_g = f['A_g'][:]
    A_c = f['A_c'][:]

    # W = f['W'][:]
    # QCQI = f['QCQI'][:]
    Z = f['Z'][:]
    Y = f['Y'][:]
    X = f['X'][:]

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)

In [ ]:
#CALLING IN TRACKED PARCELS DATA
##################################################################

In [4]:
#out_arr

# out=xr.open_dataset(dir+'tracking_algorithms/trackout/parcel_tracking_everywhere_OLD.nc')['out_arr'].values;out=out.astype(object);out[:, [0,1,2,4,5]] = out[:, [0,1,2,4,5]].astype(int) #***
# save=xr.open_dataset(dir+'tracking_algorithms/trackout/parcel_tracking_everywhere_OLD.nc')['save_arr'].values;save=save.astype(object);save[:, [0,1,2,4,5]] = save[:, [0,1,2,4,5]].astype(int) #***


out=xr.open_dataset(dir+'Project_Algorithms/Tracking_Algorithms/trackout/parcel_tracking_combined_NEW.nc')['out_arr'].values;out=out.astype(object);out[:, [0,1,2]] = out[:, [0,1,2]].astype(int) #***
save=xr.open_dataset(dir+'Project_Algorithms/Tracking_Algorithms/trackout/parcel_tracking_combined_NEW.nc')['save_arr'].values;save=save.astype(object);save[:, [0,1,2]] = save[:, [0,1,2]].astype(int) #***

out_arr=out[~np.all(out == 0, axis=1)];#print('list of first 10 SBZ parcels'); print(out_arr[:15])
save_arr=save[~np.all(save == 0, axis=1)];save_arr=save_arr[np.where(np.unique(save_arr[1:-1,0]))];#print('list of first 10 ignored parcels');print(save_arr[:5])

###############################################################################
#remove duplicates
lst=[]
unique_values, counts = np.unique(out_arr[:,0], return_counts=True); duplicates = unique_values[counts > 1]
for elem in duplicates:
    idx = np.where(out_arr[:,0] == elem)[0] 
    extras=idx[np.where(out_arr[idx,5]!=np.min(out_arr[idx,5]))]
    lst.extend([x for x in extras])
mask=np.ones(len(out_arr), dtype=bool); mask[lst] = False
out_arr=out_arr[mask]; 
###############################################################################

out_arr=out[~np.all(out == 0, axis=1)];print('list of first 10 SBZ parcels'); print(out_arr[:15])
save_arr=save[~np.all(save == 0, axis=1)];save_arr=save_arr[np.where(np.unique(save_arr[1:-1,0]))];print('list of first 10 ignored parcels');print(save_arr[:5])
placeholder=out_arr.copy(); run=True
############################################################
print(f'there are a total of {len(out_arr)} CL parcels and {len(save_arr)} nonCL parcels')


ALL_out_arr=out_arr.copy(); ALL_save_arr=save_arr.copy()

list of first 10 SBZ parcels
[[74 41 47]
 [160 20 24]
 [232 18 21]
 [244 11 14]
 [363 11 14]
 [546 7 12]
 [566 28 32]
 [746 8 12]
 [750 24 29]
 [822 22 27]
 [827 26 30]
 [874 44 49]
 [885 59 65]
 [1028 8 12]
 [1189 10 13]]
list of first 10 ignored parcels
[[18 5 10]
 [218 22 27]
 [246 7 11]
 [357 14 20]
 [414 7 11]]
there are a total of 1290 CL parcels and 1473 nonCL parcels


In [5]:
#SHALLOW

#search for deep convective parcels within lagrangian tracking output     
##############################################################
def SHALLOW_threshold(zthresh,type):

    if type=='CL':
        out_arr=ALL_out_arr.copy()
    if type=='nonCL':
        out_arr=ALL_save_arr.copy()
    
    deep_out_ind=[]; extendrange=[]
    times=data['time'].values/(1e9 * 60); times=times.astype(float);
    for ind in range(len(out_arr)): 
        #searchs if next most local max goes above zthresh
        nummins=120; numsteps=int(nummins/times[1])
        aboverange=np.arange(out_arr[ind,2],out_arr[ind,2]+numsteps,1) #range of times between current time and numsteps later
        aboverange=aboverange[aboverange<len(data['time'])] #caps out at max time
        above=parcel['z'].isel(xh=out_arr[ind,0],time=aboverange).values/1000
    
        dt=1
        #takes dabove/dt
        f=above
        ddx = (
                f[1:  ]
                -
                f[0:-1]
            ) / (
            2 * dt
        )
        signs = np.sign(ddx)
        signs_diff=np.diff(signs)
        local_maxes=np.where((signs_diff != 0) & (signs_diff < 0))[0]+1 #make sure +1 is here
        if len(local_maxes)==0:
            local_maxes=[0]
        
        if np.any(above[local_maxes[0]]<=zthresh): #< for SHALLOW, > for DEEP
            extendrange.append(local_maxes[0]) #save to extend xlim of plot later
            deep_out_ind.append(ind)
    
    out_arr=out_arr[deep_out_ind,:]
    # print(f'> {zthresh} km. {len(out_arr)} leftover parcels')
    return out_arr#, extendrange
    # print(out_arr)
##############################################################

convectivelevel=4 #4km
SHALLOW_out_arr=SHALLOW_threshold(convectivelevel,type='CL')
SHALLOW_save_arr=SHALLOW_threshold(convectivelevel,type='nonCL')

print('list of first 10 SBZ parcels'); print(out_arr[:15])
print(f'there are a total of {len(SHALLOW_out_arr)} CL parcels and {len(SHALLOW_save_arr)} nonCL parcels')


list of first 10 SBZ parcels
[[74 41 47]
 [160 20 24]
 [232 18 21]
 [244 11 14]
 [363 11 14]
 [546 7 12]
 [566 28 32]
 [746 8 12]
 [750 24 29]
 [822 22 27]
 [827 26 30]
 [874 44 49]
 [885 59 65]
 [1028 8 12]
 [1189 10 13]]
there are a total of 873 CL parcels and 1130 nonCL parcels


In [6]:
#DEEP

#search for deep convective parcels within lagrangian tracking output     
##############################################################
def DEEP_threshold(zthresh,type):
    if type=='CL':
        out_arr=ALL_out_arr.copy()
    if type=='nonCL':
        out_arr=ALL_save_arr.copy()
    
    deep_out_ind=[]; extendrange=[]
    times=data['time'].values/(1e9 * 60); times=times.astype(float);
    for ind in range(len(out_arr)): 
        #searchs if next most local max goes above zthresh
        nummins=120; numsteps=int(nummins/times[1])
        aboverange=np.arange(out_arr[ind,2],out_arr[ind,2]+numsteps,1) #range of times between current time and numsteps later
        aboverange=aboverange[aboverange<len(data['time'])] #caps out at max time
        above=parcel['z'].isel(xh=out_arr[ind,0],time=aboverange).values/1000
    
        dt=1
        #takes dabove/dt
        f=above
        ddx = (
                f[1:  ]
                -
                f[0:-1]
            ) / (
            2 * dt
        )
        signs = np.sign(ddx)
        signs_diff=np.diff(signs)
        local_maxes=np.where((signs_diff != 0) & (signs_diff < 0))[0]+1 #make sure +1 is here
        if len(local_maxes)==0:
            local_maxes=[0]
        
        if np.any(above[local_maxes[0]]>=zthresh): #< for SHALLOW, > for DEEP
            extendrange.append(local_maxes[0]) #save to extend xlim of plot later
            deep_out_ind.append(ind)
    
    out_arr=out_arr[deep_out_ind,:]
    # print(f'> {zthresh} km. {len(out_arr)} leftover parcels')
    return out_arr#, extendrange
    # print(out_arr)
##############################################################

convectivelevel=6 #4km
DEEP_out_arr=DEEP_threshold(convectivelevel,type='CL')
DEEP_save_arr=DEEP_threshold(convectivelevel,type='nonCL')

print('list of first 10 SBZ parcels'); print(out_arr[:15])
print(f'there are a total of {len(DEEP_out_arr)} CL parcels and {len(DEEP_save_arr)} nonCL parcels')

list of first 10 SBZ parcels
[[74 41 47]
 [160 20 24]
 [232 18 21]
 [244 11 14]
 [363 11 14]
 [546 7 12]
 [566 28 32]
 [746 8 12]
 [750 24 29]
 [822 22 27]
 [827 26 30]
 [874 44 49]
 [885 59 65]
 [1028 8 12]
 [1189 10 13]]
there are a total of 161 CL parcels and 167 nonCL parcels


In [ ]:
#DEFINING SOME FUNCTIONS
##################################################################

In [442]:
#VARIABLES AND GET UNITED FUNCTION
variables = data.variables
long_names = {var: {
        'long_name': data[var].attrs.get('long_name', ''),
        'units': data[var].attrs.get('units', '')
    } for var in variables}


w_vars = {var: long_name for var, long_name in long_names.items() if var.startswith('wb') or var in ['w']}
ptb_vars = {var: long_name for var, long_name in long_names.items() if var.startswith('ptb') or var in ['th']}
qvb_vars = {var: long_name for var, long_name in long_names.items() if var.startswith('qvb') or var in ['qv']}
vars = {}
vars.update(w_vars)
vars.update(ptb_vars)
vars.update(qvb_vars)

var_names=list(vars.keys())
long_names = [info['long_name'] for info in vars.values()]
units = [info['units'] for info in vars.values()]
def convert_to_latex(unit):
    parts = unit.split('/')  # Split the unit into parts
    latex_unit = parts[0]  # Start with the first part
    for part in parts[1:]:  # Iterate over the remaining parts
        latex_unit += r"\ " + part + r"^{-1}"  # Add the part with a '^{-1}' exponent
    return r"$" + latex_unit + r"$"
latex_units = [convert_to_latex(unit) for unit in units]

var_names.append("qv'")
long_names.append('perturbation water vapor mixing ratio')
latex_units.append(r'$kg\ kg^{-1}')

var_names.append("th'")
long_names.append('perturbation theta')
latex_units.append(r'$K')

def get_unit(var,var_names):
    where=np.where(np.array(var_names)==var)[0]
    final_unit=np.array(latex_units)[where][0]
    if ('wb' in var) or ('qvb' in var) or ('thb' in var) or ("'" in var):
        final_unit+=' /1000'
    return final_unit

In [460]:
#PLOTTING FUNCTION
def plot_variable(var_box_interp, var, Ng, var_names):
    def add_grid(Ng,ax):
        # #adding grid lines
        # ygrids=np.arange(0,(Ng-1),0.5)
        # xgrids=np.arange(0,(Ng-1),0.5)
        ywhich=np.linspace(0,(Ng-1),Ng+1)
        xwhich=np.linspace(0,(Ng-1),Ng+1)
       
        # ywhich=ygrids[np.where((ygrids % 1) == 0.5)]
        # xwhich=xgrids[np.where((xgrids % 1) == 0.5)]
        
        for (xind, yind) in zip(xwhich, ywhich):
            ax.axhline(yind, color='white',linestyle='--',label=None) #add grid
            ax.axvline(xind, color='white',linestyle='--',label=None) #add grid
    
    # Create figure and main gridspec (1 row, 3 columns)
    fig = plt.figure(figsize=(14, 6),constrained_layout=False)
    gs = GridSpec(1, 3, width_ratios=[2, 0.1, 1.5], figure=fig, wspace=0.3)  # Reduce space between left and middle panel
    
    # Create a 3x3 gridspec within the left panel
    gs_left = GridSpecFromSubplotSpec(3, 3, subplot_spec=gs[0, 0], wspace=0.1, hspace=0.2)
    # gs_left=gs[0].subgridspec(3, 3, wspace=0, hspace=0)
    
    # Loop over timesteps and create subplots in 3x3 format (left to right, top to bottom)
    
    if var=='qv':
        cmap='Blues'
    elif var=="qv'":
        cmap='RdBu_r'
    else:
        cmap='viridis' #cmap='RdBu_r'
    
    n_levels=30
    vmin=var_box_interp.min();vmax=var_box_interp.max()#;vmin=-vmax
    levels = np.linspace(vmin, vmax, n_levels)
    # import matplotlib.colors as mcolors
    # # norm = Normalize(vmin=vmin, vmax=vmax)
    # norm = mcolors.BoundaryNorm(boundaries=levels, ncolors=n_levels)
    # cmap = plt.get_cmap(cmap, n_levels)
    
    for t in range(9):
        row, col = divmod(t, 3)  # Get row, column indices for 3x3 layout
        ax = fig.add_subplot(gs_left[row, col])
        im = ax.contourf(var_box_interp[t],cmap=cmap,levels=levels,vmin=vmin,vmax=vmax)#,norm=norm)
        add_grid(Ng,ax)
        ax.set_title(f't = {t-1}', fontsize=10)
            
        ax.set_xticks(range(var_box_interp.shape[2]))
        ax.set_yticks(range(var_box_interp.shape[1]))    
        ax.set_xlabel('x (km)');ax.set_ylabel('y (km)')
        
        # ax.set_xticks(np.arange(-2, 3)) 
        # ax.set_yticks(np.arange(-2, 3))
        
        # Hide major tick labels selectively (not the ticks themselves)
        if col != 0:
            ax.set_yticklabels([])
            ax.set_ylabel('')
        if row != 2:
            ax.set_xticklabels([])
            ax.set_xlabel('')
    
        
    
    # Middle panel: Create a subplot for the colorbar
    ax_cbar = fig.add_subplot(gs[0, 1])  # Use the middle narrow panel
    cbar = fig.colorbar(im, cax=ax_cbar)
    cbar.ax.yaxis.set_ticks_position('left')
    # ax_cbar.set_title(label=get_unit(var,var_names), fontsize=10)
    
    # Right panel placeholder
    ax_right = fig.add_subplot(gs[0, 2])
    ax_right.set_title(f"{var} over Avg CL-LFC Trajectory")
    
    ax_right.plot(var_box_interp[:,2,2],color='black');
    ax_right.set_ylabel(get_unit(var,var_names))
    ax_right.set_xlabel(f'timesteps (5 mins)')

    fig.show()
    return fig

In [ ]:
def save_plot(fig, filename, dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/Project_Algorithms/Tracking_Algorithms/plots/'):
    # Ensure the directory exists
    if not os.path.exists(dir):
        os.makedirs(dir)

    # Construct the full file path
    file_path = os.path.join(dir, filename)

    # Save the figure to the file
    fig.savefig(file_path, dpi=300)
    print(f"plot saved")

In [ ]:
########################################################################

In [ ]:
#RUNNING FOR ONE TRAJECTORY ONLY

In [467]:
#  # t_mean=int(np.floor(np.mean(out_arr[:,2]+after+1-(out_arr[:,1]+1))))
# t_mean=9
# Nx=len(data['xh'])
# Ny=len(data['yh'])
        
# def get_var(var,t_val,z_val,y_slice,x_slice):
#     if var in  np.array(list(w_vars.keys())):
#         var_data=data[var].isel(time=t_val,
#                                 yh=y_slice, xh=x_slice).interp(zf=data['zh']).isel(zh=z_val)
#     elif var=="qv'" or var=="th'":
#         var_data=data['qv'].isel(time=t_val,zh=z_val)
#         var_data-=var_data.mean(dim=('yh','xh'))
#         var_data=var_data.isel(yh=y_slice,xh=x_slice)
#     else:
#         var_data=data[var].isel(time=t_val,zh=z_val,yh=y_slice,xh=x_slice)

#     if ('wb' in var) or ('qvb' in var) or ('thb' in var) or ("'" in var):
#         var_data*=1000
#     return var_data
    
# def get_trajectory(var,row):
#     #SETUP
#     ######
#     out_arr=ALL_out_arr
#     box_size = 2
#     Ng = 2*box_size+1
#     # after=4 #20 mins
#     after=0 #TESTING***
    
#     #GETTING INDICES
#     ################
#     p=out_arr[row,0]
#     t1=out_arr[row,1]
#     t2=out_arr[row,2]
    
#     ts=np.arange(t1-1,t2+after+1)
#     zs=Z[ts,p]
#     ys=Y[ts,p]
#     xs=X[ts,p]
    
#     #MAKING DATA
#     ############
#     var_box=np.zeros((len(ts),Ng,Ng))

#     # Skip parcels that cross the boundary
#     if np.any((ys - box_size < 0) | (ys + box_size >= Ny) | 
#           (xs - box_size < 0) | (xs + box_size >= Nx)):
#         raise ValueError('Crossed Boundary')
        
#     # Loop through each (xs, ys) pair and slice the data
#     for i in range(len(ts)):
#         # Extract the variable for the valid parcel
#         var_box[i] = get_var(
#             var,
#             ts[i],
#             zs[i], 
#             slice(ys[i] - box_size, ys[i] + box_size + 1), 
#             slice(xs[i] - box_size, xs[i] + box_size + 1)
#         )
    
#     #TIME INTERPOLATION
#     ####################
#     # Create interpolation function along axis 0 (time)
#     interp_func = interp1d(np.linspace(0, 1, var_box.shape[0]), var_box, axis=0, kind='linear', fill_value='extrapolate')
    
#     # Interpolate to new time steps
#     var_box_interp = interp_func(np.linspace(0, 1, t_mean))
    
#     return var_box,var_box_interp

# print(var_names)
# var=var_names[10]
# row=10;[var_box1,var_box_interp1]=get_trajectory(var=var,row=row)
# #PLOTTING
# fig=plot_variable(var_box_interp1, var, Ng, var_names)
# #SAVING
# save_plot(fig=fig,filename=f"Parcel_{row}_Trajectory_{var}_{res}")

In [288]:
########################################################################

In [ ]:
#RUNNING FOR MEAN OF ALL TRACKED TRAJECTORY

In [456]:
# t_mean=int(np.floor(np.mean(out_arr[:,2]+after+1-(out_arr[:,1]+1))))
t_mean=9
Nx=len(data['xh'])
Ny=len(data['yh'])
def get_var(var,t_val,z_val,y_slice,x_slice):
    if var in  np.array(list(w_vars.keys())):
        var_data=data[var].isel(time=t_val,
                                yh=y_slice, xh=x_slice).interp(zf=data['zh']).isel(zh=z_val)
    elif var=="qv'" or var=="th'":
        var_data=data['qv'].isel(time=t_val,zh=z_val)
        var_data-=var_data.mean(dim=('yh','xh'))
        var_data=var_data.isel(yh=y_slice,xh=x_slice)
    else:
        var_data=data[var].isel(time=t_val,zh=z_val,yh=y_slice,xh=x_slice)

    if ('wb' in var) or ('qvb' in var) or ('thb' in var) or ("'" in var):
        var_data*=1000
    return var_data

#SETUP
######
# out_arr=ALL_out_arr
# out_arr=SHALLOW_out_arr
out_arr=DEEP_out_arr
box_size = 2
Ng = 2*box_size+1
# after=4 #20 mins
after=0
def get_mean_trajectory(var):
    var_box_interp=np.zeros((t_mean,Ng,Ng))

    for row in np.arange(out_arr.shape[0]):
        if np.mod(row,10)==0: print(f'current row {row}')
        
        #GETTING INDICES
        ################
        p=out_arr[row,0]
        t1=out_arr[row,1]
        t2=out_arr[row,2]
        
        ts=np.arange(t1-1,t2+after+1)
        zs=Z[ts,p]
        ys=Y[ts,p]
        xs=X[ts,p]
        
        #MAKING DATA
        ############
        var_box=np.zeros((len(ts),Ng,Ng))
    
        # Skip parcels that cross the boundary
        if np.any((ys - box_size < 0) | (ys + box_size >= Ny) | 
              (xs - box_size < 0) | (xs + box_size >= Nx)):
            continue
            
        # Loop through each (xs, ys) pair and slice the data
        for i in range(len(ts)):
            # Extract the variable for the valid parcel
            var_box[i] = get_var(
                var, 
                ts[i],
                zs[i],
                slice(ys[i] - box_size, ys[i] + box_size + 1), 
                slice(xs[i] - box_size, xs[i] + box_size + 1))
            
        #TIME INTERPOLATION
        ####################
        # Create interpolation function along axis 0 (time)
        interp_func = interp1d(np.linspace(0, 1, var_box.shape[0]), var_box, axis=0, kind='linear', fill_value='extrapolate')
        
        # Interpolate to new time steps
        var_box_interp += interp_func(np.linspace(0, 1, t_mean))

    var_box_interp/=out_arr.shape[0]
    return var_box_interp

var="w"
var_box_interp2=get_mean_trajectory(var=var)
fig=plot_variable(var_box_interp2, var, Ng, var_names)
save_plot(fig=fig,filename=f"Mean_Trajectory_{var}_{res}")

# for var in var_names:
#     print(var)
#     var_box_interp2=get_mean_trajectory(var=var)
#     #PLOTTING
#     fig=plot_variable(var_box_interp2, var, Ng, var_names)
#     save_plot(fig=fig,filename=f"Mean_Trajectory_{var}_{res}")

current row 0
current row 10
current row 20
current row 30
current row 40
current row 50
current row 60
current row 70
current row 80
current row 90
current row 100
current row 110
current row 120
current row 130
current row 140
current row 150
current row 160
